# ParkVision AI — Data Preprocessing & Model Training
**SA-1 · Scenario 1: ParkVision AI (UrbanFlow AI Pvt. Ltd. / ParkSmart AI)**

Trains a MobileNetV2 slot-occupancy classifier (Occupied / Empty) on the
[PKLot dataset](https://www.inf.ufpr.br/vri/databases/PKLot.tar.gz) and
evaluates it, per Steps 2–3 of the assignment brief.

**Before running:** `Runtime -> Change runtime type -> T4 GPU` (training is
much slower on CPU).

Covers:
1. Download & inspect the PKLot dataset
2. Build a balanced, preprocessed subset (resize, augment, 70/15/15 split)
3. Train a MobileNetV2 transfer-learning classifier
4. Evaluate (accuracy, confusion matrix, classification report)
5. Save the model for use in the Streamlit app (`app.py`)


## Step 1 recap — problem definition

- **Input:** a parking-lot image (or, for training, a cropped single-slot image)
- **Output:** slot-wise status, `Occupied` or `Empty`
- **Approach chosen:** crop-based binary **classification** with MobileNetV2 (the brief's suggested classification route — simpler to get right than full detection, and PKLot already ships per-slot cropped, labeled images)
- **Known challenges:** shadows/lighting, partial occlusion, weather variation (PKLot conveniently covers Sunny/Cloudy/Rainy, which is exactly why it's the reference dataset for this scenario)

In [ ]:
!pip install -q opencv-python-headless scikit-learn kaggle

import os, json, random, shutil, zipfile
from pathlib import Path

import numpy as np
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

print("TensorFlow:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)


## Step 2 — Download the PKLot dataset

**Option A — Kaggle (recommended, fastest):**
1. Go to kaggle.com -> your profile -> *Settings* -> *API* -> *Create New Token*.
   This downloads a `kaggle.json` file.
2. Run the cell below and upload that `kaggle.json` when prompted.

**Option B — official UFPR mirror (no Kaggle account needed):**
Skip to the commented-out `wget` cell further down if you'd rather pull the
dataset directly from the paper's authors.

> The exact Kaggle dataset listing can move or get re-uploaded under a new
> slug over time — if `ammarnassanalhajali/pklot-dataset` below 404s, search
> ["PKLot dataset" on Kaggle](https://www.kaggle.com/datasets?search=pklot)
> for the current one and swap the slug in the `!kaggle datasets download`
> line.


In [ ]:
from google.colab import files

print("Upload your kaggle.json (Kaggle -> Settings -> API -> Create New Token):")
uploaded = files.upload()

os.makedirs("/root/.kaggle", exist_ok=True)
shutil.move("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)


In [ ]:
# Kaggle mirror of PKLot. If this slug no longer resolves, search Kaggle for
# "PKLot dataset" and swap it in here.
KAGGLE_SLUG = "ammarnassanalhajali/pklot-dataset"

!kaggle datasets download -d {KAGGLE_SLUG} -p /content/pklot_raw --unzip


In [ ]:
# --- Option B (alternative): official UFPR source, no Kaggle account needed ---
# Uncomment if you'd rather not use Kaggle. This is a large (~4.5GB) archive.
#
# !wget -q --show-progress https://www.inf.ufpr.br/vri/databases/PKLot.tar.gz -O /content/PKLot.tar.gz
# !mkdir -p /content/pklot_raw && tar -xzf /content/PKLot.tar.gz -C /content/pklot_raw


### Index every Occupied/Empty image

PKLot mirrors vary slightly in their exact folder layout (camera / weather /
date), but they all keep images in subfolders literally named `Occupied` and
`Empty`. Walking the tree for those folder names, rather than hardcoding an
exact path, makes this robust to whichever mirror you used above.


In [ ]:
RAW_ROOT = Path("/content/pklot_raw")

occupied_paths, empty_paths = [], []
for dirpath, dirnames, filenames in os.walk(RAW_ROOT):
    label = Path(dirpath).name.lower()
    imgs = [os.path.join(dirpath, f) for f in filenames if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    if label == "occupied":
        occupied_paths.extend(imgs)
    elif label == "empty":
        empty_paths.extend(imgs)

print(f"Found {len(occupied_paths):,} occupied images")
print(f"Found {len(empty_paths):,} empty images")
assert len(occupied_paths) > 0 and len(empty_paths) > 0, \
    "No Occupied/Empty folders found — check RAW_ROOT and the extracted dataset structure."


## Step 2 (cont.) — balanced subset, resize, split, augment

The brief asks for **at least 100 images per class**; we take 400/class here
(change `N_PER_CLASS` if you want more — PKLot has plenty of headroom) so
there's enough left after the 70/15/15 split for validation and test to be
meaningful.


In [ ]:
N_PER_CLASS = 400
IMG_SIZE = (224, 224)
OUT_ROOT = Path("/content/data")

random.shuffle(occupied_paths)
random.shuffle(empty_paths)
occupied_sample = occupied_paths[:N_PER_CLASS]
empty_sample = empty_paths[:N_PER_CLASS]

def split_paths(paths):
    train, temp = train_test_split(paths, test_size=0.30, random_state=42)
    val, test = train_test_split(temp, test_size=0.50, random_state=42)
    return {"train": train, "val": val, "test": test}

splits = {
    "occupied": split_paths(occupied_sample),
    "empty": split_paths(empty_sample),
}

for cls, parts in splits.items():
    for part, paths in parts.items():
        print(f"{cls:>8} / {part:<5}: {len(paths)} images")


In [ ]:
def resize_and_copy(src_paths, dest_dir):
    dest_dir.mkdir(parents=True, exist_ok=True)
    for i, src in enumerate(src_paths):
        img = cv2.imread(src)
        if img is None:
            continue
        img = cv2.resize(img, IMG_SIZE)
        cv2.imwrite(str(dest_dir / f"{i:05d}.jpg"), img)

for cls, parts in splits.items():
    for part, paths in parts.items():
        resize_and_copy(paths, OUT_ROOT / part / cls)

print("Done. Folder layout:")
!find /content/data -maxdepth 2 -type d | sort


### Quick sanity-check EDA

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for col, cls in enumerate(["occupied", "empty"]):
    sample_dir = OUT_ROOT / "train" / cls
    sample_files = sorted(os.listdir(sample_dir))[:5]
    for row, fname in enumerate(sample_files):
        img = cv2.cvtColor(cv2.imread(str(sample_dir / fname)), cv2.COLOR_BGR2RGB)
        axes[row, col].imshow(img)
        axes[row, col].set_title(cls if row == 0 else "")
        axes[row, col].axis("off")
plt.tight_layout()
plt.show()


## Step 3 — Model Development\n\n### Data pipelines (with augmentation)

In [ ]:
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    OUT_ROOT / "train", image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="binary", seed=42
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    OUT_ROOT / "val", image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="binary", seed=42
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    OUT_ROOT / "test", image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="binary", shuffle=False
)

class_names = train_ds.class_names  # alphabetical: ['empty', 'occupied'] -> occupied = label 1
print("Class order (label 0 / 1):", class_names)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomBrightness(0.15),
])

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y)).prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)


### Build the MobileNetV2 transfer-learning model

In [ ]:
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet"
)
base_model.trainable = False  # freeze for the first training pass

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = preprocess_input(inputs)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
x = tf.keras.layers.Dense(128, activation="relu")(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)
model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
model.summary()


### Train (15–30 epochs per the brief, with early stopping)

In [ ]:
EPOCHS = 25

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint("best_model.keras", monitor="val_accuracy", save_best_only=True),
]

history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["accuracy"], label="train")
axes[0].plot(history.history["val_accuracy"], label="val")
axes[0].set_title("Accuracy"); axes[0].legend()
axes[1].plot(history.history["loss"], label="train")
axes[1].plot(history.history["val_loss"], label="val")
axes[1].set_title("Loss"); axes[1].legend()
plt.show()


## Step 3 (cont.) — Evaluation on the held-out test set

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test accuracy: {test_acc:.4f}")

y_true = np.concatenate([y.numpy() for _, y in test_ds]).astype(int).ravel()
y_prob = model.predict(test_ds).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(cmap="Blues")
plt.title("Confusion matrix — test set")
plt.show()


### Optional: fine-tuning for a higher-accuracy / "Distinguished" rubric score

Unfreezes the top layers of MobileNetV2 and re-trains briefly at a much
lower learning rate. Skip this cell if your frozen-base accuracy already
looks solid and you're short on Colab GPU time.


In [ ]:
base_model.trainable = True
# keep the early, more generic layers frozen; only fine-tune the later ones
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

fine_tune_history = model.fit(
    train_ds, validation_data=val_ds, epochs=10, callbacks=callbacks
)

test_loss, test_acc = model.evaluate(test_ds)
print(f"Test accuracy after fine-tuning: {test_acc:.4f}")


## Save the model for the Streamlit app

In [ ]:
MODEL_OUT = "parking_classifier.keras"
model.save(MODEL_OUT)
print(f"Saved {MODEL_OUT}")

# Download it, then drop it into parkvision-ai/model/parking_classifier.keras
# in your project repo (create the `model/` folder if it doesn't exist).
from google.colab import files
files.download(MODEL_OUT)


## Step 4 preview — feeding predictions into the parking-insight logic

The congestion-level and recommendation rules (Step 4 of the brief) live in
`utils/insights.py` in the Streamlit project so the dashboard and this
notebook use the *exact same* logic — no risk of the two drifting apart.
Upload that one file here (or `pip install` nothing needed, it's dependency-free)
to sanity-check a batch of predictions end-to-end before moving to the app.


In [ ]:
from google.colab import files
print("Upload utils/insights.py from the project (optional, for this demo cell):")
uploaded = files.upload()


In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location("insights", "insights.py")
insights = importlib.util.module_from_spec(spec)
spec.loader.exec_module(insights)

# Treat one batch of test-set predictions as if they were N parking slots
demo_results = [
    insights.SlotResult(slot_id=i, occupied=bool(p), confidence=float(prob if p else 1 - prob))
    for i, (p, prob) in enumerate(zip(y_pred[:20], y_prob[:20]))
]
summary = insights.summarize_lot(demo_results)
print(summary)


## Next steps

1. **Step 5 (Streamlit app):** `app.py` in the project repo already does the
   slot-crop -> classify -> overlay -> insights pipeline — just drop your
   saved `parking_classifier.keras` into its `model/` folder.
2. **Define real slot coordinates:** run `slot_position_picker.py` on a
   reference photo of the lot you'll demo with, to produce a
   `slot_positions.json` for `app.py` (the bundled demo one is synthetic).
3. **Step 6 (testing):** try `app.py` against unseen lot photos — different
   lighting/weather/angle — and note failure cases in the README.
4. **Step 7 (deployment):** push the repo to GitHub and deploy on
   [Streamlit Cloud](https://streamlit.io/cloud). See the project README for
   the full checklist.
